### Agents with Tools (Function Calling)

[Documentation Link](https://strandsagents.com/docs/user-guide/concepts/tools/)

#### Importing required libraries

In [41]:
from strands import Agent, tool, ToolContext
from strands.models.ollama import OllamaModel
from strands_tools import calculator # pre-defined tools from strands
from pydantic import BaseModel, Field
 


#### Configure local Ollama model

In [78]:
# let's configure local ollama model - not all parameters are required - adjust as per need
model = OllamaModel(
    host="http://localhost:11434",  # ollama runs on this / The address of the Ollama server
    model_id="qwen3:30b",         # The Ollama model identifier - choose the model as per your hardware configuration
    temperature=1,                # Controls randomness (higher = more random)
    max_tokens=2048,                # Maximum number of tokens to generate
    #keep_alive="10m",               # How long the model stays loaded in memory
    top_p=0.8                      # Controls diversity via nucleus sampling
    #stop_sequences=["###", "END"],  # List of sequences that stop generation
    #options={"top_k": 40}           # Additional model parameters (e.g., top_k)
)

#### Initialization and invocation of agent

##### Agents with strands inbuilt tools

In [104]:
# let's create system_prompt to give basic instrcution to the agent

system_prompt = """
## ROLE:
- You are user friendly agent to answer user queries.
- You must answer only from the given tools; `calculator`

## GUIDELINES MUST FOLLOW:
- If user's question is not related to the available tools then you must reply:
> Sorry I could not answer to this question.
- You must not try to answer by yourself even you know the answer. Must use given tool to answer.
- Do not include interanl steps in your answer.

## OUTPUT FORMAT:
- Must be short and summarize answer.
- Do not add any comentory.
- Your response must be simple text.
- Do not include interanl steps in your answer.

"""

In [97]:
# let's initialize the agent with above model config along with the in-built tools
agent = Agent(model=model,
              system_prompt=system_prompt,
              tools=[calculator])

In [ ]:
# # as we using small models then let's use structure output
# class AgentResponse(BaseModel):
#     answer:str = Field(..., description="Short and summarize answer with respect to the user's question. If answer not found then must be 'Sorry I could not answer to this question.'")

In [81]:
response  = agent("What is difference between Assistive AI and Agentic AI?")

Sorry I could not answer to this question.

In [82]:
print(str(response)) # must be str(response) as AgentResult exposes str for their response

Sorry I could not answer to this question.



In [83]:
# actual response resided here
response.message["content"][0]["text"]

'Sorry I could not answer to this question.'

In [84]:
# checking execution metrics
response.metrics

EventLoopMetrics(cycle_count=1, tool_metrics={}, cycle_durations=[17.276514291763306], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='76e22b2f-43cd-4cfd-8b6b-87ff50a26b9b', usage={'inputTokens': 1332, 'outputTokens': 142, 'totalTokens': 1474})], usage={'inputTokens': 1332, 'outputTokens': 142, 'totalTokens': 1474})], traces=[<strands.telemetry.metrics.Trace object at 0x000002C913C132F0>], accumulated_usage={'inputTokens': 1332, 'outputTokens': 142, 'totalTokens': 1474}, accumulated_metrics={'latencyMs': 16814})

In [85]:
# checking execution metrics of tool call happened
response.metrics.tool_metrics

{}

In above we could see no tool called heppend

In [98]:
response2  = agent("What is 2 * 2")


Tool #1: calculator


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ 2 * 2               │                                                                            │
│  │ Result    │ 4                   │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

4

In [99]:
print(str(response2))

4



In [101]:
response2.metrics.tool_metrics

{'calculator': ToolMetrics(tool={'toolUseId': 'tooluse_d77cb099624046ac9badc958', 'name': 'calculator', 'input': {'expression': '2 * 2', 'mode': 'evaluate'}}, call_count=1, success_count=1, error_count=0, total_time=0.0043659210205078125)}

In above we could see tool called heppend.

Note: It is common if above metrics is blank, as we know this is SLM model and we are running locally. If we have 70b or more model then we can expect good behavior.

##### Agents with custom tools

In [102]:
# temp db
user_details = {
    "1":{"name": "Simran", "dep": "train", "salary": 5000},
    "2":{"name": "Kartik", "dep": "it", "salary": 6000},
    "3":{"name": "John", "dep": "cyber", "salary": 12000},
    "4":{"name": "Rock", "dep": "wwe", "salary": 7000},
    "5":{"name": "batman", "dep": "public", "salary": 0},
    "6":{"name": "valak", "dep": "horror", "salary": 9000},
    "7":{"name": "rajni", "dep": "movie", "salary": 13000}
}

Documentation link for [strands state](https://strandsagents.com/docs/user-guide/concepts/agents/state/)

We could also use storage to persist agent state, refer this [documentation](https://strandsagents.com/docs/user-guide/concepts/storage/) However we have to use mechanism to tell what to store.

In [103]:
# defining custom tools

@tool
def get_user_details(user_id:str) -> dict | str:
    """
    Tool to get the user details from the database for the given user id.

    Args:
        user_id: id of the user for which details need to be fetched

    Returns:
        dict | str: return dict if user details found otherwise str
    """
    return user_details.get(user_id, "No record found")



# let's add tracking like who is asking this details

@tool(context=True)
def get_all_user_details(tool_context: ToolContext) -> dict:
    """
    Tool to get the all users details from the database to address user query.

    Returns:
        dict: return dict as all user details
    """
    print(f"Query asked by: {tool_context.agent.state.get('current_user_id', 'no current user id')}")
    return user_details 


In [112]:
# let's create system_prompt to give basic instrcution to the agent

system_prompt_wct = """
## ROLE:
- You are user friendly agent to answer user queries.
- You must answer only from the given tools; `get_user_details` and `get_all_user_details`

## GUIDELINES MUST FOLLOW:
- If user's question is not related to the available tools then you must reply:
    > Sorry I could not answer to this question.
- You must not try to answer by yourself even you know the answer. Must use given tool to answer.
- Do not include interanl steps in your answer.
- If no relevant info found must reply:
    > No relevant information found.

## OUTPUT FORMAT:
- Must be short and summarize answer. Do not overwelm the user. 
- Answer must be clear and short.
- Do not add any comentory.
- Your response must be simple text.
- Do not include interanl steps in your answer.

"""

In [113]:
# let's define custom tool to agent

# let's initialize the agent with above model config along with the in-built tools
agent_wct = Agent(model=model,
              system_prompt=system_prompt_wct,
              tools=[get_user_details, get_all_user_details],
              state={"current_user_id": "10"})

In [114]:
wct_response  = agent_wct("Can you tell me the salary of user 5?")


Tool #1: get_user_details
0

In [115]:
print(str(wct_response))
print(wct_response.metrics.tool_metrics)

0

{'get_user_details': ToolMetrics(tool={'toolUseId': 'tooluse_cbbd33a409294cd0aa580bc9', 'name': 'get_user_details', 'input': {'user_id': '5'}}, call_count=1, success_count=1, error_count=0, total_time=0.0004968643188476562)}


Wow, we could see the answer is correct and tool call happened to get the data. let's ask the same question again and see if agent does tool call again or just refer it's previous messages. 

Note: When we work production ready code we do not maintain process alive, thus we have to use external source to store these messages.

In [116]:
wct_response2  = agent_wct("Can you tell me the salary of user 5?")

0

In [117]:
print(str(wct_response2))
print(wct_response2.metrics.tool_metrics)

0

{'get_user_details': ToolMetrics(tool={'toolUseId': 'tooluse_cbbd33a409294cd0aa580bc9', 'name': 'get_user_details', 'input': {'user_id': '5'}}, call_count=1, success_count=1, error_count=0, total_time=0.0004968643188476562)}


Okay, nice it seems it is following system prompt. But no surprises if it does not call and refer from previous messages.

In [118]:
print(str(agent_wct("what is the user name of having user id as 19?")))



Tool #2: get_user_details
No relevant information found.No relevant information found.



In [119]:
# let's ask from another tool, it should also print curent user id as 10
wct_response3 = agent_wct("which user has heighest salary?")


Tool #3: get_all_user_details
No relevant information found.

In [120]:
print(str(wct_response3))
print(wct_response3.metrics.tool_metrics)

No relevant information found.

{'get_user_details': ToolMetrics(tool={'toolUseId': 'tooluse_b36502f14cb14d038219744a', 'name': 'get_user_details', 'input': {'user_id': '19'}}, call_count=2, success_count=2, error_count=0, total_time=0.0010259151458740234), 'get_all_user_details': ToolMetrics(tool={'toolUseId': 'tooluse_fbd0ce0659c84b448e3a7854', 'name': 'get_all_user_details', 'input': {}}, call_count=1, success_count=0, error_count=1, total_time=0.000675201416015625)}


hmmm, strange it did call correct tool but was unable to give the details. Fine. if we give more context and refine the prompt we can get the work doen from agent.